# Inputs: prompts, tools, and data

A JAZ agent runs a Python REPL: the model writes code, JAZ runs it, the model sees the result and keeps going until it returns.

Everything you give that agent goes in the same way — as a **keyword argument**. This tutorial is about that one idea, and nothing else.

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [1]:
%pip install -q jaz-lang

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [3]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## One kind of argument

**JAZ has no notion of a “prompt” argument, a “tool” argument or a “data” argument.** There is no such distinction to make, because none is needed.

Every keyword argument you pass to `invoke` is an arbitrary Python object that:

1. becomes a **variable in the agent's REPL**, under the name you gave it, and
2. is **described in the agent's prompt**, so the model knows it is there.

That is the whole input model. No schemas, no registration, no serialization.

To see it, we need to look at the prompt — so first a small helper. **Skip past the class itself**; you are not expected to understand it yet.

In [4]:
from jaz import invoke
from jaz.hooks import Hook


class PrintUserPrompt(Hook):
    """Prints the prompt the agent is given, so we can see it.

    You do NOT need to understand this class to follow this tutorial. It is scaffolding:
    it lets us look at what the model was handed. Hooks are the subject of tutorial 3.
    """

    def __init__(self):
        self.printed = False

    def on_llm_query_enter(self, event):
        if not self.printed:
            for message in event.messages:
                if message["role"] == "user":
                    print(message["content"])
                    self.printed = True
                    break
        return []

Now a call with two inputs: a string and a list.

In [5]:
with PrintUserPrompt():
    answer = invoke(
        task="Add the numbers together.",
        numbers=[3, 8, 14],
    )

answer

<task type="str">
Add the numbers together.
</task>
<numbers type="list">
[3, 8, 14]
</numbers>
The input variables `task`, `numbers` are available in your REPL with descriptions given above.
They are not automatically available to `invoke()` calls you make, so making them available to sub-invokes requires passing them explicitly.



25

Read the prompt above. `task` and `numbers` are listed the **same way**, each with its type. Nothing marks one as the instruction and the other as the data — the agent simply received two named values and worked out what to do.

`task` is not a keyword JAZ knows. It is a string that happens to be called `task`; `question` or `instructions` would behave identically.

## A tool is just an input you can call

Nothing new is needed for tools. Pass the function.

The example below matters because **the model cannot answer it on its own** — the stock levels live in a dict here. Notice the task never says which function to call, or when.

In [6]:
# A tiny "database". The model has no way to know what is in it.
INVENTORY = {"SKU-1": 12, "SKU-2": 0, "SKU-3": 7}


def stock_level(sku: str) -> int:
    """Units currently in stock for a SKU. Raises KeyError for an unknown SKU."""
    return INVENTORY[sku]


with PrintUserPrompt():
    reply = invoke(
        task="Is SKU-2 in stock? Answer yes or no.",
        stock_level=stock_level,
    )

reply

<task type="str">
Is SKU-2 in stock? Answer yes or no.
</task>
<stock_level type="function">
`stock_level(sku: str) -> int`: Units currently in stock for a SKU. Raises KeyError for an unknown SKU.
</stock_level>
The input variables `task`, `stock_level` are available in your REPL with descriptions given above.
They are not automatically available to `invoke()` calls you make, so making them available to sub-invokes requires passing them explicitly.



'no'

Look at how `stock_level` was described: its **signature and docstring**. That is why a tool needs no schema — a Python function already carries its own description, and the agent decided by itself to call it.

## How an input is described

What the model is shown depends on what the object is:

| The input is… | The model is shown |
| :------------ | :----------------- |
| anything else (the default) | its `str()` |
| a class | its docstring and public members |
| any other callable — a function, a method, an object with `__call__` | its signature and docstring |
| an object with no `str()` of its own | its class docstring and public attributes |

You have already seen two of those rows: `numbers` printed as `[3, 8, 14]`, and `stock_level` as a signature.

Here is the last row — an object that does not define its own `str()`:

In [7]:
from jaz import describe


class Ledger:
    def __init__(self, rows):
        self.rows = rows


ledger = Ledger([("a", 1), ("b", 2)])

# Without a description of its own, an object renders as its class and public attributes.
with PrintUserPrompt():
    invoke(task="Say OK.", ledger=ledger)

<task type="str">
Say OK.
</task>
<ledger type="Ledger">
`ledger` [Ledger]: (no description available)
  - `rows` [list]
</ledger>
The input variables `task`, `ledger` are available in your REPL with descriptions given above.
They are not automatically available to `invoke()` calls you make, so making them available to sub-invokes requires passing them explicitly.



Accurate, but not very useful. `describe` replaces the default rendering with your own, and travels with the object:

In [8]:
described = describe(ledger, "ledger: 2 rows of (name, amount) pairs.")

with PrintUserPrompt():
    invoke(task="Say OK.", ledger=described)

<task type="str">
Say OK.
</task>
<ledger type="Ledger">
ledger: 2 rows of (name, amount) pairs.
</ledger>
The input variables `task`, `ledger` are available in your REPL with descriptions given above.
They are not automatically available to `invoke()` calls you make, so making them available to sub-invokes requires passing them explicitly.



## What to take away

- **One mechanism.** A prompt, a tool, a number, a list, your own object — all keyword arguments, all variables in the REPL, all described in the prompt.
- **`task` is a convention**, not a keyword.
- **Tools need no registration.** A function's signature and docstring are its interface, and the agent chooses when to call it.
- **`describe`** overrides how an input is rendered when the default is wrong.

---

**Next:** [Recursive subagents and scope](02-subagents-and-scope.ipynb) — an agent can call `invoke` itself, which raises the question of what a *subagent* can see.